# Incident Composition Archetypes

## 1) Problem framing
**Business question:** Are there recurring incident profiles that need different prevention playbooks?

- Explanatory goal: quantify severity relationships by incident and context attributes (including resident history *before* the incident).
- Predictive goal: multiclass incident bucket + a **binary auxiliary** task with enough support (high severity / safety-critical types).
- Success metrics: explanatory R2/MAE on severity; predictive macro-F1; binary ROC-AUC / F1.

**Feature policy:** Prior-window counts, entropy, and house–type rates use only incidents **strictly before** the current row time for that resident (no future leakage). Smoothed house–type rates use dataset-wide Laplace smoothing for a stable prior (documented in code).

Deployment candidate: `/api/reports/trends/incident-archetypes` and incident prevention planning widget.


In [ ]:

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    roc_auc_score,
    r2_score,
)
from sklearn.model_selection import GroupShuffleSplit, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent, Path.cwd().parent / 'ml-pipelines']:
    if (candidate / 'data_loader.py').exists():
        sys.path.insert(0, str(candidate))
        break
from data_loader import load_table

inc = load_table('incident_reports').sort_values('incident_date').copy()
res = load_table('residents')[
    ['resident_id', 'current_risk_level', 'case_category', 'referral_source']
]
df = inc.merge(res, on='resident_id', how='left')
severity_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
df.loc[:, 'severity_num'] = df['severity'].map(severity_map).fillna(2)
top_types = df['incident_type'].value_counts().head(4).index
df.loc[:, 'incident_bucket'] = np.where(df['incident_type'].isin(top_types), df['incident_type'], 'Other')
df.loc[:, 'month'] = df['incident_date'].dt.month
df.loc[:, 'dayofweek'] = df['incident_date'].dt.dayofweek

# Smoothed P(type | safehouse) — global prior; alpha avoids zeros
alpha = 1.0
type_house = df.groupby(['safehouse_id', 'incident_type']).size().reset_index(name='tc')
house_n = df.groupby('safehouse_id').size().reset_index(name='hn')
n_types = df['incident_type'].nunique()
type_house = type_house.merge(house_n, on='safehouse_id')
type_house['smoothed_rate'] = (type_house['tc'] + alpha) / (type_house['hn'] + alpha * n_types)
df = df.merge(
    type_house[['safehouse_id', 'incident_type', 'smoothed_rate']],
    on=['safehouse_id', 'incident_type'],
    how='left',
)

def shannon_entropy(counts: np.ndarray) -> float:
    p = counts[counts > 0] / counts.sum()
    return float(-(p * np.log(p + 1e-12)).sum()) if len(p) else 0.0

def enrich_resident_history(sub: pd.DataFrame) -> pd.DataFrame:
    sub = sub.sort_values('incident_date').copy()
    prior_30 = []
    prior_90 = []
    days_since = []
    prior_max_sev = []
    ent_90 = []
    for i, row in sub.iterrows():
        t = row['incident_date']
        past = sub[sub['incident_date'] < t]
        p30 = past[past['incident_date'] >= t - pd.Timedelta(days=30)]
        p90 = past[past['incident_date'] >= t - pd.Timedelta(days=90)]
        prior_30.append(len(p30))
        prior_90.append(len(p90))
        if len(past):
            days_since.append((t - past['incident_date'].max()).days)
            prior_max_sev.append(float(past['severity_num'].max()))
            vc = p90['incident_type'].value_counts()
            ent_90.append(shannon_entropy(vc.values.astype(float)))
        else:
            days_since.append(365)
            prior_max_sev.append(1.0)
            ent_90.append(0.0)
    sub.loc[:, 'prior_count_30d'] = prior_30
    sub.loc[:, 'prior_count_90d'] = prior_90
    sub.loc[:, 'days_since_last_inc'] = days_since
    sub.loc[:, 'prior_max_severity'] = prior_max_sev
    sub.loc[:, 'type_entropy_90d'] = ent_90
    return sub

parts = [enrich_resident_history(g) for _, g in df.groupby('resident_id', sort=False)]
df = pd.concat(parts, ignore_index=True)

# Binary auxiliary: critical safety / escalation (High or Critical)
df.loc[:, 'y_binary_high_sev'] = (df['severity_num'] >= 3).astype(int)
# Alternative slice: runaway / behavioral / self-harm cluster
risk_types = {'RunawayAttempt', 'Behavioral', 'SelfHarm', 'Security'}
df.loc[:, 'y_binary_risk_cluster'] = df['incident_type'].isin(risk_types).astype(int)

features = [
    'safehouse_id', 'resolved', 'follow_up_required', 'month', 'dayofweek',
    'current_risk_level', 'reported_by', 'case_category', 'referral_source',
    'prior_count_30d', 'prior_count_90d', 'days_since_last_inc',
    'prior_max_severity', 'type_entropy_90d', 'smoothed_rate',
]
X = df[features].copy()
y_reg = df['severity_num']
y_clf = df['incident_bucket']
y_bin = df['y_binary_high_sev']

cat_cols = [c for c in X.columns if X[c].dtype == 'object' or str(X[c].dtype) == 'bool']
num_cols = [c for c in X.columns if c not in cat_cols]
prep = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

# Group-aware holdout: residents
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_tr, idx_te = next(gss.split(X, y_clf, groups=df['resident_id']))
Xtr, Xte = X.iloc[idx_tr], X.iloc[idx_te]
ytr_reg, yte_reg = y_reg.iloc[idx_tr], y_reg.iloc[idx_te]
ytrc, ytec = y_clf.iloc[idx_tr], y_clf.iloc[idx_te]
ytrb, yteb = y_bin.iloc[idx_tr], y_bin.iloc[idx_te]

lin = Pipeline([('prep', prep), ('model', LinearRegression())])
lin.fit(Xtr, ytr_reg)
pred_reg = lin.predict(Xte)
print('Explanatory R2:', round(r2_score(yte_reg, pred_reg), 3))
print('Explanatory MAE:', round(mean_absolute_error(yte_reg, pred_reg), 3))

rf = Pipeline([('prep', prep), ('model', RandomForestClassifier(
    n_estimators=250, random_state=42, min_samples_leaf=3, class_weight='balanced_subsample'))])
rf.fit(Xtr, ytrc)
pred = rf.predict(Xte)
print('Predictive accuracy:', round(accuracy_score(ytec, pred), 3))
print('Predictive macro-F1:', round(f1_score(ytec, pred, average='macro', zero_division=0), 3))

cv = cross_validate(rf, X, y_clf, cv=5, scoring=['accuracy', 'f1_macro'], n_jobs=-1)
print('CV accuracy mean/std:', round(float(cv['test_accuracy'].mean()), 3), round(float(cv['test_accuracy'].std()), 3))

# Binary auxiliary
rf_b = Pipeline([('prep', prep), ('model', RandomForestClassifier(
    n_estimators=250, random_state=42, min_samples_leaf=3, class_weight='balanced'))])
rf_b.fit(Xtr, ytrb)
if yteb.nunique() >= 2:
    proba_b = rf_b.predict_proba(Xte)[:, 1]
    pred_b = (proba_b >= 0.5).astype(int)
    print('Binary (High/Critical) holdout AUC:', round(roc_auc_score(yteb, proba_b), 3))
    print('Binary (High/Critical) holdout F1:', round(f1_score(yteb, pred_b, zero_division=0), 3))
else:
    print('Binary track skipped: single class in holdout.')

coef_values = np.ravel(lin.named_steps['model'].coef_)
coef_names = lin.named_steps['prep'].get_feature_names_out()
usable = min(len(coef_values), len(coef_names))
coef = pd.Series(coef_values[:usable], index=coef_names[:usable]).sort_values(key=lambda s: s.abs(), ascending=False).head(12)
print('\nTop explanatory effects:')
print(coef.round(4).to_string())

print('\nDecision recommendations: pair high prior-incident load and rising type entropy with targeted de-escalation and staffing for the dominant archetypes.')


In [ ]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent, Path.cwd().parent / 'ml-pipelines']:
    if (candidate / 'trend_eval_helpers.py').exists():
        sys.path.insert(0, str(candidate))
        break
from trend_eval_helpers import print_calibration_bins, print_threshold_scan, bootstrap_linear_coefs, fairness_binary, fairness_regression_mae

print("\n=== Evaluation artifacts ===")
if y_bin.nunique() >= 2 and yteb.nunique() >= 2:
    proba = rf_b.predict_proba(Xte)[:, 1]
    print("\n--- Binary High/Critical: calibration ---")
    print_calibration_bins(yteb.values, proba)
    print("\n--- Threshold scan ---")
    print_threshold_scan(yteb.values, proba)
    fairness_binary(rf_b, Xte, yteb, df.loc[Xte.index], ['safehouse_id', 'case_category'], min_n=10)
bootstrap_linear_coefs(lin, Xtr, ytr_reg, n_boot=150, top_k=8)
